# LeetCode 3389 — Minimum Operations to Make Character Frequencies Equal bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-character-frequencies-equal/
- Độ khó: **Hard**
- Chủ đề: **Đếm tần số**, **duyệt mọi tần số đích `t`**, **quy hoạch động theo chữ cái** với nhận xét "chuyển chỉ cần giữa hai chữ kề nhau, không nối chuỗi"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def makeStringGood(self, s: str) -> int:
```

## 1. Mục tiêu học

1. Bỏ thứ tự ký tự, chỉ giữ mảng tần số `occ[0..25]`; ba thao tác trở thành: `occ[x] += 1`, `occ[x] −= 1`, hoặc chuyển một đơn vị `occ[x] → occ[x+1]`.
2. Xâu tốt ⇔ mọi `occ[x]` bằng `0` hoặc bằng cùng một hằng `t` (`t ≥ 1`) ⇒ duyệt `t`.
3. Với `t` cố định, mỗi chữ chọn đích `0` hoặc `t`; thao tác "chuyển" chỉ có lợi khi chữ `x` **thừa** và chữ `x + 1` **thiếu** — gộp thành chi phí `max(thừa, thiếu)`.
4. DP `f[i]` trên chữ cái (từ `z` về `a`) với hai lựa chọn: xử lý `i` riêng, hoặc ghép `i` với `i + 1`.
5. Kiểm chứng bằng **BFS** trên không gian vectơ tần số cho dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Xâu **tốt** nếu mọi ký tự (có mặt) xuất hiện cùng số lần. Thao tác: xóa một ký tự, chèn một ký tự, hoặc đổi một ký tự thành ký tự **kế tiếp** trong bảng chữ cái (không đổi `z → a`). Tìm số thao tác ít nhất để `s` tốt.

**Ví dụ 1:** `"acab"` → `1`. **Ví dụ 2:** `"wddw"` → `0`. **Ví dụ 3:** `"aaabc"` → `2`.

**Ràng buộc:** `3 ≤ n ≤ 2·10^4`.

In [1]:
from collections import Counter, deque

examples = [("acab", 1), ("wddw", 0), ("aaabc", 2)]
for s, exp in examples:
    print(repr(s), dict(sorted(Counter(s).items())), '-> kỳ vọng', exp)

'acab' {'a': 2, 'b': 1, 'c': 1} -> kỳ vọng 1
'wddw' {'d': 2, 'w': 2} -> kỳ vọng 0
'aaabc' {'a': 3, 'b': 1, 'c': 1} -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Thao tác trên mảng tần số

Thứ tự ký tự không ảnh hưởng tính "tốt" và chi phí ⇒ chỉ cần `occ`. Xóa/chèn ký tự `x`: `occ[x] ∓ 1`. Đổi `x → x+1`: `occ[x] −= 1, occ[x+1] += 1`.

In [2]:
def occ_of(s):
    o = [0] * 26
    for ch in s:
        o[ord(ch) - 97] += 1
    return o

def is_good_occ(o):
    nz = {c for c in o if c}
    return len(nz) <= 1

print(occ_of("aaabc")[:3], is_good_occ(occ_of("wddw")), is_good_occ(occ_of("acab")))

[3, 1, 1] True False


### 3.2. Hai nhận xét (theo gợi ý đề)

1. **Không bao giờ cần chuyển qua hai bước** `x → x+1 → x+2`: thay bằng "xóa ở `x`, chèn ở `x+2`" cũng tốn 2. Vậy mỗi đơn vị chỉ chuyển tối đa **một** bước.
2. **Không bao giờ vừa tăng vừa giảm** cùng một `occ[x]`.

Hệ quả: với `t` cố định, một đơn vị được chuyển `x → x+1` chỉ có lợi nếu `x` đang **thừa** (so với đích của nó) và `x + 1` đang **thiếu** (đích `t`, `occ[x+1] < t`); khi đó chữ `x + 1` không còn thừa để chuyển tiếp.

### 3.3. Chi phí ghép một cặp

Nếu chữ `x` thừa `e` đơn vị và chữ `x+1` thiếu `d` đơn vị: chuyển `min(e, d)` đơn vị (mỗi đơn vị 1 thao tác), phần thừa còn lại xóa, phần thiếu còn lại chèn ⇒ tổng `min(e, d) + (e − min) + (d − min) = max(e, d)`.

In [3]:
for e, d in [(3, 1), (1, 3), (2, 2), (0, 4)]:
    print(f"thừa {e}, thiếu {d}: không ghép {e + d}, ghép {max(e, d)}")

thừa 3, thiếu 1: không ghép 4, ghép 3
thừa 1, thiếu 3: không ghép 4, ghép 3
thừa 2, thiếu 2: không ghép 4, ghép 2
thừa 0, thiếu 4: không ghép 4, ghép 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3 `"aaabc"`: `occ(a, b, c) = (3, 1, 1)`.

- `t = 1`: xóa 2 chữ `a` ⇒ 2.
- `t = 2`: `a` thừa 1, `b` thiếu 1 ⇒ ghép tốn `max(1, 1) = 1`; `c` thiếu 1 ⇒ chèn 1 ⇒ tổng 2.
- `t = 3`: `b`, `c` thiếu 2 mỗi chữ (hoặc xóa) ⇒ ≥ 2.

Đáp án 2.

Một ví dụ mà thao tác "đổi" giúp rõ: `"aaaabb"` (`4, 2`), `t = 3`: đổi một `a` thành `b` ⇒ `(3, 3)` — **1** thao tác; chỉ xóa/chèn thì cần 2 (ví dụ xóa hai chữ `a`).

## 5. Cách 1 — Brute Force: BFS trên vectơ tần số

Chỉ giữ các chữ từ `min(s)` đến `max(s)` (chữ ngoài khoảng không cần đụng tới… với BFS nhỏ ta giới hạn như vậy để không gian trạng thái nhỏ) và giới hạn mỗi tần số ≤ `2n + 2`. Dùng cho xâu ngắn trên 2–3 chữ cái.

In [4]:
def make_good_bfs(s):
    lo, hi = ord(min(s)), ord(max(s))
    m = hi - lo + 1
    occ = [0] * m
    for ch in s:
        occ[ord(ch) - lo] += 1
    CAP = 2 * len(s) + 2
    start = tuple(occ)
    dist = {start: 0}
    dq = deque([start])
    while dq:
        st = dq.popleft()
        d = dist[st]
        nz = {c for c in st if c}
        if len(nz) == 1:
            return d
        for i in range(m):
            for delta in (1, -1):
                v = st[i] + delta
                if 0 <= v <= CAP:
                    ns = st[:i] + (v,) + st[i + 1:]
                    if ns not in dist:
                        dist[ns] = d + 1; dq.append(ns)
            if i + 1 < m and st[i] > 0:
                ns = list(st); ns[i] -= 1; ns[i + 1] += 1; ns = tuple(ns)
                if ns not in dist:
                    dist[ns] = d + 1; dq.append(ns)

for s, exp in examples:
    print(make_good_bfs(s), '| kỳ vọng', exp)

1 | kỳ vọng 1
0 | kỳ vọng 0
2 | kỳ vọng 2


**Độ phức tạp:** số trạng thái `~(2n)^m` — chỉ cho `n ≤ 8`, `m ≤ 3`. (Việc chỉ xét các chữ trong `[min(s), max(s)]` đủ cho kiểm chứng: chuyển sang chữ lớn hơn `max(s)` hoặc chèn chữ mới không bao giờ có lợi hơn — xem mục 10.)

## 6. Ý tưởng thuật toán / bước đột phá

Với mỗi `t ∈ [1, max(occ)]`, tính `f[i]` = chi phí nhỏ nhất xử lý các chữ `i..25` khi **không** có đơn vị nào chuyển vào chữ `i`:

- **Riêng:** `f[i] = min(occ[i], |occ[i] − t|) + f[i+1]` (đích `0` hoặc `t`).
- **Ghép với `i + 1`** (khi `occ[i+1] < t`, `d = t − occ[i+1]`): chữ `i` chọn đích `0` (thừa `e = occ[i]`) hoặc đích `t` nếu `occ[i] ≥ t` (thừa `e = occ[i] − t`); chi phí `max(e, d) + f[i+2]`.

Đáp án `min_t f_t[0]`. Mỗi `t` tốn `O(26)` ⇒ tổng `O(26 · max occ)` ≈ `5·10^5`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP với lượng chuyển **tường minh** (kiểm chứng nhận xét)

Trạng thái `(i, carry)`: số đơn vị chuyển từ chữ `i − 1` vào chữ `i`. Với mỗi `t`, chữ `i` (đã nhận `carry`) chọn đích `0` hoặc `t` và chọn số đơn vị `k` chuyển tiếp sang `i + 1` (chi phí `k`, lấy từ phần thừa hiện có), phần thừa còn lại xóa, phần thiếu chèn. Không áp đặt nhận xét 3.2 nên là kiểm chứng độc lập, nhưng `O(26 · C²)` cho mỗi `t` ⇒ chỉ cho dữ liệu nhỏ.

In [5]:
def make_good_carry(s):
    occ = occ_of(s)
    best = float('inf')
    for t in range(1, max(occ) + 1):
        C = max(occ) + t + 1
        INF = float('inf')
        cur = {0: 0}                                  # carry -> chi phí
        for i in range(26):
            nxt = {}
            for carry, cost in cur.items():
                have = occ[i] + carry
                for target in (0, t):
                    surplus = max(0, have - target)
                    deficit = max(0, target - have)
                    for k in range(0, (surplus if i < 25 else 0) + 1):   # chuyển k sang i+1
                        c = cost + k + (surplus - k) + deficit
                        if c < nxt.get(k, INF):
                            nxt[k] = c
            cur = nxt
        best = min(best, cur.get(0, INF))
    return best

for s, exp in examples:
    print(make_good_carry(s), '| kỳ vọng', exp)

1 | kỳ vọng 1
0 | kỳ vọng 0
2 | kỳ vọng 2


### 7.2. Cách 3 — DP ghép cặp kề (lời giải chính)

In [6]:
def make_good_dp(s):
    occ = occ_of(s)
    best = float('inf')
    for t in range(1, max(occ) + 1):
        f = [0] * 28
        for i in range(25, -1, -1):
            c = occ[i]
            res = min(c, abs(c - t)) + f[i + 1]
            if i + 1 < 26 and occ[i + 1] < t:
                d = t - occ[i + 1]
                res = min(res, max(c, d) + f[i + 2])               # i về 0, chuyển phần thừa
                if c >= t:
                    res = min(res, max(c - t, d) + f[i + 2])       # i giữ t, chuyển phần dư
            f[i] = res
        best = min(best, f[0])
    return best

for s, exp in examples:
    print(make_good_dp(s), '| kỳ vọng', exp)

1 | kỳ vọng 1
0 | kỳ vọng 0
2 | kỳ vọng 2


### 7.3. Cách 4 — Cùng công thức, viết **top-down có nhớ**

`g(i)` = chi phí nhỏ nhất cho các chữ `i..25` khi không có gì chuyển vào chữ `i`. Cách viết đệ quy bám sát lập luận "xử lý riêng hoặc ghép với chữ kế tiếp", dùng để đối chiếu bản lặp.

In [7]:
from functools import lru_cache

def make_good_memo(s):
    occ = tuple(occ_of(s))
    best = float('inf')
    for t in range(1, max(occ) + 1):
        @lru_cache(maxsize=None)
        def g(i):
            if i >= 26:
                return 0
            c = occ[i]
            res = min(c, abs(c - t)) + g(i + 1)
            if i + 1 < 26 and occ[i + 1] < t:
                d = t - occ[i + 1]
                res = min(res, max(c, d) + g(i + 2))
                if c >= t:
                    res = min(res, max(c - t, d) + g(i + 2))
            return res
        best = min(best, g(0))
    return best

for s, exp in examples:
    print(make_good_memo(s), '| kỳ vọng', exp)

1 | kỳ vọng 1
0 | kỳ vọng 0
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 3 trên `"aaaabb"` với từng `t`.

In [8]:
def trace(s):
    occ = occ_of(s)
    letters = [i for i in range(26) if occ[i]]
    print('tần số:', {chr(97 + i): occ[i] for i in letters})
    for t in range(1, max(occ) + 1):
        f = [0] * 28
        choice = [''] * 26
        for i in range(25, -1, -1):
            c = occ[i]
            res, why = min(c, abs(c - t)) + f[i + 1], 'riêng'
            if i + 1 < 26 and occ[i + 1] < t:
                d = t - occ[i + 1]
                if max(c, d) + f[i + 2] < res:
                    res, why = max(c, d) + f[i + 2], f'ghép ({chr(97 + i)}->0, chuyển sang {chr(98 + i)})'
                if c >= t and max(c - t, d) + f[i + 2] < res:
                    res, why = max(c - t, d) + f[i + 2], f'ghép ({chr(97 + i)}->t, chuyển dư sang {chr(98 + i)})'
            f[i] = res; choice[i] = why
        print(f"t = {t}: f[a] = {f[0]}; lựa chọn ở các chữ có mặt: {[(chr(97 + i), choice[i]) for i in letters]}")
    print('đáp án:', make_good_dp(s))

trace("aaaabb")

tần số: {'a': 4, 'b': 2}
t = 1: f[a] = 4; lựa chọn ở các chữ có mặt: [('a', 'riêng'), ('b', 'riêng')]
t = 2: f[a] = 2; lựa chọn ở các chữ có mặt: [('a', 'riêng'), ('b', 'riêng')]
t = 3: f[a] = 1; lựa chọn ở các chữ có mặt: [('a', 'ghép (a->t, chuyển dư sang b)'), ('b', 'riêng')]
t = 4: f[a] = 2; lựa chọn ở các chữ có mặt: [('a', 'riêng'), ('b', 'riêng')]
đáp án: 1


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def makeStringGood(self, s: str) -> int:
        occ = [0] * 26
        for ch in s:
            occ[ord(ch) - 97] += 1
        best = len(s)
        for t in range(1, max(occ) + 1):
            f = [0] * 28
            for i in range(25, -1, -1):
                c = occ[i]
                res = (c if c < abs(c - t) else abs(c - t)) + f[i + 1]
                if i < 25 and occ[i + 1] < t:
                    d = t - occ[i + 1]
                    v = (c if c > d else d) + f[i + 2]
                    if v < res:
                        res = v
                    if c >= t:
                        e = c - t
                        v = (e if e > d else d) + f[i + 2]
                        if v < res:
                            res = v
                f[i] = res
            if f[0] < best:
                best = f[0]
        return best

for s, exp in examples:
    got = Solution().makeStringGood(s)
    print(got, 'OK' if got == exp else 'SAI')

1 OK
0 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Đích:** xâu tốt ⇔ tồn tại `t ≥ 1` để mọi `occ[x] ∈ {0, t}`; duyệt mọi `t ≤ max(occ)` là đủ (với `t > max(occ)` mọi chữ có mặt đều thiếu, đưa về `t = max(occ)` rẻ hơn).
2. **Cấu trúc lời giải tối ưu (3.2):** mỗi đơn vị chuyển tối đa một bước; chữ nhận đơn vị chuyển vào thì không chuyển ra (nếu không ta có chuyển hai bước, hoặc vừa tăng vừa giảm). Vì vậy các "cặp chuyển" `(x, x+1)` rời nhau, và chữ `x + 1` trong cặp có đích `t` với `occ[x+1] < t` (nếu không, chuyển vào rồi lại xóa là vô ích).
3. **Chi phí cặp (3.3):** `max(e, d)`; chi phí riêng: `min(occ, |occ − t|)`. DP chọn tập cặp rời nhau tối ưu ⇒ đúng.
4. Cách 2 không dùng nhận xét mà vẫn cho cùng kết quả (mục 12) — kiểm chứng thực nghiệm cho các nhận xét.

## 11. Phân tích Time / Space Complexity

`M = max(occ) ≤ n`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS vectơ tần số | mũ theo số chữ | lớn |
| 2. DP chuyển tường minh | `O(M · 26 · M²)` | `O(M)` |
| 3. DP ghép cặp | `O(26 · M)` | `O(26)` |
| 4. Top-down có nhớ | `O(26 · M)` | `O(26)` + đệ quy |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'carry': make_good_carry, 'dp': make_good_dp, 'memo': make_good_memo,
         'submit': lambda s: Solution().makeStringGood(s)}
edge = [("aaa", 0), ("aab", 1), ("aaab", 1), ("abc", 0), ("aaac", 1), ("zzzy", 1), ("abbccc", 1)]
for s, exp in examples + edge:
    assert make_good_bfs(s) == exp, (s, make_good_bfs(s))
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS (2–3 chữ cái) và giữa các cách

In [11]:
import random
rng = random.Random(3389)
cnt = 0
for _ in range(250):
    s = ''.join(rng.choice(rng.choice(['ab', 'abc', 'ac'])) for _ in range(rng.randint(3, 7)))
    exp = make_good_bfs(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
for _ in range(200):
    s = ''.join(rng.choice('abcdefg') for _ in range(rng.randint(3, 40)))
    assert make_good_carry(s) == make_good_dp(s) == Solution().makeStringGood(s)
print('Khớp BFS trên', cnt, 'xâu ngắn; DP tường minh = DP ghép cặp trên 200 xâu dài hơn.')

Khớp BFS trên 250 xâu ngắn; DP tường minh = DP ghép cặp trên 200 xâu dài hơn.


### 12.3. Ca lớn `n = 2·10^4`

In [12]:
import time
n = 2 * 10 ** 4
cases = {
    'ngẫu nhiên 26 chữ': ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n)),
    'toàn a': 'a' * n,
    'a nhiều, các chữ khác ít': 'a' * (n - 25) + 'bcdefghijklmnopqrstuvwxyz',
    'tần số tăng dần': ''.join(chr(97 + i) * (i * 50) for i in range(26))[:n],
}
for name, s in cases.items():
    t = time.time(); r = Solution().makeStringGood(s); t1 = time.time() - t
    print(f"{name:26s}: {r:6d} | {t1:.2f}s")

ngẫu nhiên 26 chữ         :    525 | 0.00s
toàn a                    :      0 | 0.14s


a nhiều, các chữ khác ít  :     25 | 0.14s
tần số tăng dần           :   4700 | 0.01s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua thao tác "đổi sang chữ kế tiếp" (chỉ xóa/chèn)

In [13]:
def wrong_no_change(s):
    occ = occ_of(s)
    return min(sum(min(c, abs(c - t)) for c in occ) for t in range(1, max(occ) + 1))

print('sai :', wrong_no_change("aaaabb"), '| đúng:', Solution().makeStringGood("aaaabb"))

sai : 2 | đúng: 1


**Vì sao sai:** `"aaaabb" → "aaabbb"` chỉ cần đổi một `a` thành `b` (1 thao tác); chỉ xóa/chèn thì mọi `t` đều tốn ít nhất 2.

### Lỗi 2 — Cho phép đổi sang **bất kỳ** chữ với chi phí 1

In [14]:
def wrong_any_change(s):
    occ = occ_of(s)
    best = float('inf')
    for t in range(1, max(occ) + 1):
        surplus = deficit = 0
        cost_alone = 0
        for c in occ:
            if c == 0:
                continue
            if c >= t:
                surplus += c - t
            else:
                # chọn bỏ hẳn hay bù cho đủ t
                if c <= t - c:
                    surplus += c
                else:
                    deficit += t - c
        best = min(best, max(surplus, deficit))        # SAI: coi mọi chuyển đều tốn 1
    return best

print('sai :', wrong_any_change("aaaacc"), '| đúng:', Solution().makeStringGood("aaaacc"))

sai : 1 | đúng: 2


**Vì sao sai:** `"aaaacc"`: bản sai nghĩ đổi một `a` thành `c` tốn 1 (ra `a:3, c:3`), nhưng `a → c` cần **hai** lần đổi (`a → b → c`); chỉ chữ **kế tiếp** mới tốn 1. Đáp án đúng là 2. Lời giải đúng chỉ cho phép ghép hai chữ kề nhau.

### Lỗi 3 — Chỉ thử các `t` bằng một tần số đang có

In [15]:
def wrong_t_only_existing(s):
    occ = occ_of(s)
    best = float('inf')
    for t in set(c for c in occ if c):
        f = [0] * 28
        for i in range(25, -1, -1):
            c = occ[i]
            res = min(c, abs(c - t)) + f[i + 1]
            if i + 1 < 26 and occ[i + 1] < t:
                d = t - occ[i + 1]
                res = min(res, max(c, d) + f[i + 2])
                if c >= t:
                    res = min(res, max(c - t, d) + f[i + 2])
            f[i] = res
        best = min(best, f[0])
    return best

rng = random.Random(3)
ce = next(s for s in (''.join(rng.choice('abc') for _ in range(rng.randint(3, 9))) for _ in range(20000))
          if wrong_t_only_existing(s) != make_good_dp(s))
print('phản ví dụ:', ce, occ_of(ce)[:3], '| sai:', wrong_t_only_existing(ce), '| đúng:', make_good_dp(ce))

phản ví dụ: abaaab [4, 2, 0] | sai: 2 | đúng: 1


**Vì sao sai:** tần số đích tối ưu có thể là giá trị "ở giữa" không trùng tần số nào (ví dụ hai chữ có tần số 1 và 3 cùng về 2).

### Lỗi 4 — Khi ghép cặp chỉ xét trường hợp chữ `i` về `0`

In [16]:
def wrong_pair_only_zero(s):
    occ = occ_of(s)
    best = float('inf')
    for t in range(1, max(occ) + 1):
        f = [0] * 28
        for i in range(25, -1, -1):
            c = occ[i]
            res = min(c, abs(c - t)) + f[i + 1]
            if i + 1 < 26 and occ[i + 1] < t:
                res = min(res, max(c, t - occ[i + 1]) + f[i + 2])   # SAI: thiếu nhánh "i giữ t, chuyển phần dư"
            f[i] = res
        best = min(best, f[0])
    return best

print('sai :', wrong_pair_only_zero("aaaabb"), '| đúng:', Solution().makeStringGood("aaaabb"))

sai : 2 | đúng: 1


**Vì sao sai:** `"aaaabb"` (`a:4, b:2`), `t = 3`: chữ `a` **giữ** 3 và chuyển phần dư 1 sang `b` (thiếu 1) — tổng 1 thao tác. Nếu chỉ cho `a` về 0 khi ghép thì phải tốn `max(4, 1) = 4`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp lại không quan trọng" ⇒ chỉ làm việc với mảng tần số.
- "Mọi giá trị bằng 0 hoặc cùng một hằng" ⇒ duyệt hằng `t`.
- Thao tác "chuyển sang phần tử kề" ⇒ chứng minh chỉ cần chuyển một bước ⇒ DP ghép cặp kề như "domino".
- Kiểm chứng nhận xét bằng một DP/BFS tổng quát hơn trên dữ liệu nhỏ.

## 15. Biến thể của bài

1. Cho phép đổi `z → a` (vòng) ⇒ DP trên vòng: thử hai trường hợp cho cặp `(z, a)`.
2. Cho phép đổi sang chữ **trước** cũng được ⇒ cặp `(x, x+1)` theo cả hai chiều: chi phí cặp `max(e, d)` với vai trò thừa/thiếu đổi chỗ.
3. Yêu cầu **đúng `k` chữ khác nhau** ⇒ thêm chiều đếm số chữ có đích `t`.
4. Bài liên quan: LC 1647 (Minimum Deletions to Make Character Frequencies Unique).

In [17]:
# Biến thể 2: cho phép đổi sang chữ trước hoặc sau (cả hai chiều), DP ghép cặp hai chiều; đối chiếu BFS
def make_good_two_way(s):
    occ = occ_of(s)
    best = float('inf')
    for t in range(1, max(occ) + 1):
        f = [0] * 28
        for i in range(25, -1, -1):
            c = occ[i]
            res = min(c, abs(c - t)) + f[i + 1]
            if i + 1 < 26:
                c2 = occ[i + 1]
                opts_i = [(0, c, 0)] + ([(t, c - t, 0)] if c >= t else [(t, 0, t - c)])    # (đích, thừa, thiếu)
                opts_j = [(0, c2, 0)] + ([(t, c2 - t, 0)] if c2 >= t else [(t, 0, t - c2)])
                for _, ei, di in opts_i:
                    for _, ej, dj in opts_j:
                        # chuyển từ bên thừa sang bên thiếu (một chiều có tác dụng)
                        cost = max(ei, dj) + max(ej, di) if (ei and dj) or (ej and di) else ei + di + ej + dj
                        res = min(res, cost + f[i + 2])
            f[i] = res
        best = min(best, f[0])
    return best

def bfs_two_way(s):
    lo, hi = ord(min(s)), ord(max(s)); m = hi - lo + 1
    occ = [0] * m
    for ch in s:
        occ[ord(ch) - lo] += 1
    CAP = 2 * len(s) + 2
    start = tuple(occ); dist = {start: 0}; dq = deque([start])
    while dq:
        st = dq.popleft(); d = dist[st]
        if len({c for c in st if c}) == 1:
            return d
        moves = []
        for i in range(m):
            for delta in (1, -1):
                if 0 <= st[i] + delta <= CAP:
                    moves.append(st[:i] + (st[i] + delta,) + st[i + 1:])
            for j in (i - 1, i + 1):
                if 0 <= j < m and st[i] > 0:
                    ns = list(st); ns[i] -= 1; ns[j] += 1; moves.append(tuple(ns))
        for ns in moves:
            if ns not in dist:
                dist[ns] = d + 1; dq.append(ns)

rng = random.Random(15)
for _ in range(150):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(3, 7)))
    assert make_good_two_way(s) == bfs_two_way(s), (s, make_good_two_way(s), bfs_two_way(s))
print('Biến thể hai chiều khớp BFS: "abbb" ->', make_good_two_way("abbb"))

Biến thể hai chiều khớp BFS: "abbb" -> 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | BFS vectơ tần số | mũ | Oracle |
| 2 | DP chuyển tường minh | `O(26 M³)` | Kiểm chứng nhận xét |
| 3 | DP ghép cặp kề | `O(26 M)` | **Nộp bài** |
| 4 | Top-down có nhớ | `O(26 M)` | Cùng công thức |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"abcabc"`.

**Bài 2 (tính tay):** `"aaaabb"`.

**Bài 3 (điền TODO):** viết `cost_for_t(s, t)` — chi phí nhỏ nhất với tần số đích `t`.

**Bài 4 (tự viết lại):** truy vết và dựng một **xâu tốt** cụ thể đạt chi phí tối thiểu.

**Bài 5 (mini challenge):** biến thể 1 — cho phép đổi `z → a`.

In [18]:
def cost_for_t_todo(s, t):
    # TODO
    raise NotImplementedError

def build_good_string_todo(s):
    # TODO
    raise NotImplementedError

def make_good_circular_todo(s):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi chữ xuất hiện 2 lần. (2) Đã tốt. (3) `0`.

**Bài 2.** (1) `a:4, b:2`. (2) `t = 3`: đổi một `a` thành `b` ⇒ `a:3, b:3`. (3) `1`.

**Bài 3.** (1) Tách vòng lặp trong của cách 3. (2) Trả `f[0]`. (3) Kiểm tra `min_t` bằng đáp án.

**Bài 4.** (1) Lưu lựa chọn ở mỗi `i`. (2) Dựng `occ` mới theo lựa chọn. (3) Ghép các chữ thành xâu và đếm thao tác.

**Bài 5.** (1) Thêm khả năng cặp `(z, a)`. (2) Thử "cặp `(z, a)` không dùng" (DP thường) và "dùng" (cố định `z`, `a` rồi DP trên `b..y`). (3) Đối chiếu BFS vòng.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().makeStringGood("abcabc") == 0 and Solution().makeStringGood("aaaabb") == 1 == make_good_bfs("aaaabb")

def cost_for_t(s, t):
    occ = occ_of(s)
    f = [0] * 28
    for i in range(25, -1, -1):
        c = occ[i]
        res = min(c, abs(c - t)) + f[i + 1]
        if i + 1 < 26 and occ[i + 1] < t:
            d = t - occ[i + 1]
            res = min(res, max(c, d) + f[i + 2])
            if c >= t:
                res = min(res, max(c - t, d) + f[i + 2])
        f[i] = res
    return f[0]

rng = random.Random(3)
for _ in range(300):
    s = ''.join(rng.choice('abcde') for _ in range(rng.randint(3, 30)))
    assert min(cost_for_t(s, t) for t in range(1, max(occ_of(s)) + 1)) == Solution().makeStringGood(s)
print('Bài 1: 0 | Bài 2: 1 | Bài 3 khớp.')

Bài 1: 0 | Bài 2: 1 | Bài 3 khớp.


In [20]:
# Bài 4, 5
def build_good_string(s):
    occ = occ_of(s)
    best = (float('inf'), None, None)
    for t in range(1, max(occ) + 1):
        f = [0] * 28; ch = [None] * 26
        for i in range(25, -1, -1):
            c = occ[i]
            res, pick = min(c, abs(c - t)) + f[i + 1], ('alone',)
            if i + 1 < 26 and occ[i + 1] < t:
                d = t - occ[i + 1]
                if max(c, d) + f[i + 2] < res:
                    res, pick = max(c, d) + f[i + 2], ('pair', 0)
                if c >= t and max(c - t, d) + f[i + 2] < res:
                    res, pick = max(c - t, d) + f[i + 2], ('pair', t)
            f[i], ch[i] = res, pick
        if f[0] < best[0]:
            best = (f[0], t, ch)
    cost, t, ch = best
    new = [0] * 26
    i = 0
    while i < 26:
        if ch[i][0] == 'alone':
            new[i] = 0 if occ[i] <= abs(occ[i] - t) else t
            i += 1
        else:
            new[i], new[i + 1] = ch[i][1], t
            i += 2
    return cost, ''.join(chr(97 + k) * new[k] for k in range(26))

def make_good_circular(s):
    # cặp (z, a): z thừa chuyển sang a thiếu. Thử: không dùng cặp vòng (DP thường), hoặc dùng (z, a) rồi DP trên b..y
    occ = occ_of(s)
    best = Solution().makeStringGood(s)
    for t in range(1, max(occ) + 1):
        if occ[0] < t:
            d = t - occ[0]
            for e in ([occ[25]] + ([occ[25] - t] if occ[25] >= t else [])):
                f = [0] * 28
                for i in range(24, 0, -1):
                    c = occ[i]
                    res = min(c, abs(c - t)) + f[i + 1]
                    if i + 1 < 25 and occ[i + 1] < t:
                        dd = t - occ[i + 1]
                        res = min(res, max(c, dd) + f[i + 2])
                        if c >= t:
                            res = min(res, max(c - t, dd) + f[i + 2])
                    f[i] = res
                best = min(best, max(e, d) + f[1])
    return best

rng = random.Random(5)
for _ in range(300):
    s = ''.join(rng.choice('abcdef') for _ in range(rng.randint(3, 30)))
    cost, t = build_good_string(s)
    assert cost == Solution().makeStringGood(s) and is_good_occ(occ_of(t))
print('Bài 4: dựng được xâu tốt, ví dụ "aaabc" ->', build_good_string("aaabc"))
print('Bài 5: "zzza" (cho phép z->a):', make_good_circular("zzza"), '| không vòng:', Solution().makeStringGood("zzza"))

Bài 4: dựng được xâu tốt, ví dụ "aaabc" -> (2, 'abc')
Bài 5: "zzza" (cho phép z->a): 1 | không vòng: 1


Lưu ý: xâu dựng ở bài 4 chỉ là **một** xâu tốt có tần số tối ưu; vì thứ tự ký tự không quan trọng nên ta in theo thứ tự bảng chữ cái.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for t in 1..max(occ):
    f[26] = f[27] = 0
    for i = 25..0:
        f[i] = min(occ[i], |occ[i]-t|) + f[i+1]
        if occ[i+1] < t:  d = t - occ[i+1]
            f[i] = min(f[i], max(occ[i], d) + f[i+2])               # i -> 0
            if occ[i] >= t: f[i] = min(f[i], max(occ[i]-t, d) + f[i+2])   # i -> t
    ans = min(ans, f[0])
```

**Checklist:**

- [ ] Có dùng thao tác "đổi sang chữ kế tiếp" (ghép cặp kề)?
- [ ] Chỉ ghép chữ `i` với `i + 1` (không bắc cầu, không vòng `z → a`)?
- [ ] Duyệt **mọi** `t` từ 1 tới `max(occ)`?
- [ ] Khi ghép, xét cả hai nhánh: chữ `i` về `0` hoặc giữ `t` (chuyển phần dư)?
- [ ] Mỗi chữ chọn đích `0` hoặc `t`?